# SentinelFL — complete experiment study (shared dataset adapter)

This notebook finishes the research part of SentinelFL on the **real** data in Google Drive:

* **One dataset adapter for every model.** `federated/sentinelfl` aligns raw CIC IoT 2023 data to the
  39-feature / 7-class contract; centralized training, FedAvg, local-only training, XGBoost and the
  FastAPI backend all go through it.
* **Fair comparison.** Same cleaned cache, same split files as the existing runs, same federated
  log1p scaler, same initial weights, same training function, same number of passes over the data,
  same checkpoint rule (**best validation macro-F1**).
* **Test set touched once.** Each selected checkpoint is evaluated a single time on its scenario's
  held-out test split, after training has finished.
* **Resumable.** Every finished run writes `result.json`; re-running a cell skips finished work, so a
  Colab disconnect only loses the run in progress.

Nothing in this notebook overwrites the existing `cache/`, `splits/` or `runs/` folders.
Results go to a new folder, `FL_Project/results_v2/`.

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE    = '/content/drive/MyDrive/Mini Project CSY'
FL      = f'{BASE}/FL_Project'
DATA    = f'{BASE}/CICBOTNET'          # raw CSVs (only needed for the cache check)
CACHE   = f'{FL}/cache'
SPLITS  = f'{FL}/splits'
OUT     = f'{FL}/results_v2'           # NEW folder — existing runs are left untouched
for p in (CACHE, SPLITS):
    assert os.path.isdir(p), f'missing: {p}'
print(sorted(os.listdir(SPLITS)))

## 2. Get the code

Option A (after the `feature/shared-adapter-pipeline` branch is pushed to GitHub): clone it.
Option B: upload `MiniProjectBotnet_shared_adapter.zip` to `FL_Project/` and unzip it.

In [ ]:
import subprocess
REPO = '/content/MiniProjectBotnet'
ZIP  = f'{FL}/MiniProjectBotnet_shared_adapter.zip'
if not os.path.isdir(REPO):
    if os.path.exists(ZIP):
        subprocess.run(['unzip', '-q', ZIP, '-d', '/content/'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '-b', 'feature/shared-adapter-pipeline',
                        'https://github.com/veervishal-dubey/MiniProjectBotnet.git', REPO], check=True)
import sys
sys.path.insert(0, f'{REPO}/federated')
import sentinelfl, fl_core
print('sentinelfl', sentinelfl.__version__, '| features', len(sentinelfl.FEATURES), '| classes', sentinelfl.CLASSES)

## 3. Unit tests (≈30 s) — code correctness only (synthetic data)

In [ ]:
!pip -q install xgboost pytest
!cd {REPO} && python -m pytest federated/tests -q

## 4. Check that the existing cache is exactly what the shared adapter produces (≈2–4 min)

Rebuilds the cache **in memory** from the raw CSVs listed in `cache/clean_meta.json` through
`CICIoT2023Adapter` + `clean_for_training`, and compares it with `X_clean.npy` / `y_clean.npy`.
`ok: True` means every existing split and result was produced by the same pipeline as the adapter.

In [ ]:
from sentinelfl.cache import verify_cache
verify_cache(DATA, CACHE)

## 5. Smoke run on real data (≈3 min, subsampled — NOT a result)
Checks the plumbing on the real cache before the long runs. Written to a separate folder.

In [ ]:
from sentinelfl.experiments import run_grid, summarize
run_grid(CACHE, SPLITS, f'{FL}/results_smoke', ['A_iid'], [42], ['fedavg', 'centralized'],
         rounds=2, max_train_per_client=3000, max_val_per_client=500, max_test_rows=20000)

## 6. Reproduction check: new runner vs the original Flower run (≈15 min)

Seed 42, A_iid, 20 rounds, 1 local epoch, the same seed formula as FLipynb cell 7. The original Flower
run reached validation macro-F1 **0.9840 / 0.9845** at round 20 (two runs). Agreement within normal CPU
non-determinism confirms the plain-PyTorch FedAvg matches the Flower implementation.

In [ ]:
import json, pandas as pd
run_grid(CACHE, SPLITS, OUT, ['A_iid'], [42], ['fedavg'])
r = json.load(open(f'{OUT}/runs/A_iid_s42/fedavg/result.json'))
new = pd.DataFrame(r['history'])[['round', 'val_accuracy', 'val_macro_f1']]
old = pd.read_csv(f'{FL}/runs/A_iid_s42_R20_E1/round_history.csv')[['round', 'fed_val_acc', 'fed_val_macroF1']]
cmp_ = new.merge(old, on='round'); cmp_['diff_f1'] = cmp_.val_macro_f1 - cmp_.fed_val_macroF1
display(cmp_.round(4).tail(8))

## 7. Main study — Phase 1: seed 42, all scenarios

`fedavg`, `centralized` (same 20 passes), `xgboost` (corrected 7-class reference). Approximate CPU times
on a standard Colab runtime: FedAvg ≈ 15 min, centralized ≈ 10 min, XGBoost ≈ 5–10 min per scenario.

In [ ]:
SCENARIOS = ['A_iid', 'A_dir0.5', 'A_dir0.1', 'B_filehold_iid']
run_grid(CACHE, SPLITS, OUT, SCENARIOS, [42], ['fedavg', 'centralized', 'xgboost'])

## 8. Non-IID analysis: local-only clients (≈15 min per scenario)
Each client trains alone on its own data and is evaluated on the shared test set. Shows what a client
gains by joining the federation, especially under α = 0.1.

In [ ]:
run_grid(CACHE, SPLITS, OUT, ['A_dir0.5', 'A_dir0.1'], [42], ['local'])

## 9. Phase 2: extra seeds for mean ± std (run when you have time; resumable)

In [ ]:
run_grid(CACHE, SPLITS, OUT, SCENARIOS, [43, 44], ['fedavg', 'centralized'])

## 10. Results

In [ ]:
from IPython.display import Markdown
res = summarize(OUT, log_fn=lambda *_: None)
display(Markdown(res['markdown']))

In [ ]:
import matplotlib.pyplot as plt, glob
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(5 * len(SCENARIOS), 3.6), sharey=True)
for ax, sc in zip(axes, SCENARIOS):
    for m, style in (('fedavg', 'o-'), ('centralized', 's--')):
        p = f'{OUT}/runs/{sc}_s42/{m}/result.json'
        if os.path.exists(p):
            h = pd.DataFrame(json.load(open(p))['history'])
            ax.plot(h.iloc[:, 0], h['val_macro_f1'], style, ms=3, label=m)
    ax.set_title(sc); ax.set_xlabel('round / epoch (passes over data)'); ax.grid(alpha=.3)
axes[0].set_ylabel('validation macro-F1'); axes[0].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/convergence_s42.png', dpi=130); plt.show()

In [ ]:
# Per-client validation macro-F1 of the selected FedAvg model (client-specific class imbalance)
rows = []
for sc in SCENARIOS:
    p = f'{OUT}/runs/{sc}_s42/fedavg/result.json'
    if os.path.exists(p):
        for c in json.load(open(p))['client_val_at_selection']:
            rows.append({'scenario': sc, 'client': c['client'], 'n_val': c['n_val'], 'macro_f1': c['macro_f1']})
display(pd.DataFrame(rows).pivot(index='client', columns='scenario', values='macro_f1').round(4))

## 11. Publish results to the web app

* `benchmarks.json` → `experiments/results/` in the repo (served by `GET /api/v1/evaluation/benchmarks`).
* A selected FedAvg bundle (weights **+ adapter + federated scaler**) → `ml/bundles/`, served by setting
  `ACTIVE_MODEL_BUNDLE=ml/bundles/fedavg-A_iid-s42` for the backend.

Commit those files from your machine (Colab cannot push without credentials).

In [ ]:
import shutil
!python {REPO}/scripts/run_experiments.py --out "{OUT}" --summarize-only --publish-to "{FL}/publish/experiments/results"
src = f'{OUT}/runs/A_iid_s42/fedavg/bundle'
if os.path.isdir(src):
    shutil.copytree(src, f'{FL}/publish/ml/bundles/fedavg-A_iid-s42', dirs_exist_ok=True)
print('Copy the contents of', f'{FL}/publish', 'into the repository root and commit.')